In [7]:
from datasets import load_dataset
import re
import json
import os
import random
import torch
import numpy as np
from typing import List
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    Adafactor,
    EarlyStoppingCallback,
    get_scheduler
)
import evaluate

In [2]:
train_dataset = load_dataset("LabHC/bias_in_bios", split='train')
test_dataset = load_dataset("LabHC/bias_in_bios", split='test')
dev_dataset = load_dataset("LabHC/bias_in_bios", split='dev')

In [3]:
train_dataset[0]

{'hard_text': 'He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates',
 'profession': 21,
 'gender': 0}

In [4]:
def neutralize_text(text: str) -> str:
    """
    Converts gendered pronouns and corresponding verb forms to gender-neutral ones.
    
    This function performs the following substitutions:
      - "he"/"she" → "they" (with proper casing)
      - "his"/"her" → "their"
      - "him"/"her" → "them"
      - "himself"/"herself" → "themselves"
      - When these pronouns are immediately followed by a verb that requires plural conjugation,
        e.g. "is" → "are", "was" → "were", "has" → "have".
      - It also handles versions when the pronouns are enclosed in square brackets.
    
    Args:
      text (str): The input text with gendered language.
      
    Returns:
      str: The text after gender-neutral transformations.
    """
    
    # --- Handle pronoun + verb patterns first ---
    # For "he is" or "she is" → "they are"
    text = re.sub(
        r"\b([Hh]e|[Ss]he)\s+is\b",
        lambda m: ("They are" if m.group(1)[0].isupper() else "they are"),
        text
    )
    # For "he was" or "she was" → "they were"
    text = re.sub(
        r"\b([Hh]e|[Ss]he)\s+was\b",
        lambda m: ("They were" if m.group(1)[0].isupper() else "they were"),
        text
    )
    # For "he has" or "she has" → "they have"
    text = re.sub(
        r"\b([Hh]e|[Ss]he)\s+has\b",
        lambda m: ("They have" if m.group(1)[0].isupper() else "they have"),
        text
    )
    
    # --- Handle standalone pronoun substitutions ---
    # Replace "he" or "she" (unbracketed)
    text = re.sub(
        r"\b([Hh]e|[Ss]he)\b",
        lambda m: ("They" if m.group(1)[0].isupper() else "they"),
        text
    )
    # Replace possessive adjectives: "his" or "her" → "their"
    text = re.sub(
        r"\b([Hh]is|[Hh]er)\b",
        lambda m: ("Their" if m.group(1)[0].isupper() else "their"),
        text
    )
    # Replace object pronouns: "him" or "her" → "them"
    text = re.sub(
        r"\b([Hh]im|[Hh]er)\b",
        lambda m: ("Them" if m.group(1)[0].isupper() else "them"),
        text
    )
    # Replace reflexive pronouns: "himself" or "herself" → "themselves"
    text = re.sub(
        r"\b([Hh]imself|[Hh]erself)\b",
        lambda m: ("Themselves" if m.group(1)[0].isupper() else "themselves"),
        text
    )
    
    # --- Handle bracketed pronoun cases (if applicable) ---
    # For bracketed pronouns, e.g., "[he]" → "[they]"
    text = re.sub(
        r"\[([Hh]e|[Ss]he)\]",
        lambda m: ("[They]" if m.group(1)[0].isupper() else "[they]"),
        text
    )
    text = re.sub(
        r"\[([Hh]is|[Hh]er)\]",
        lambda m: ("[Their]" if m.group(1)[0].isupper() else "[their]"),
        text
    )
    text = re.sub(
        r"\[([Hh]im|[Hh]er)\]",
        lambda m: ("[Them]" if m.group(1)[0].isupper() else "[them]"),
        text
    )
    text = re.sub(
        r"\[([Hh]imself|[Hh]erself)\]",
        lambda m: ("[Themselves]" if m.group(1)[0].isupper() else "[themselves]"),
        text
    )
    
    # --- Additional adjustments (optional) ---
    # If you encounter contractions or negated forms like "he isn't" or "she hasn't",
    # you may want to expand them first or add extra rules.
    #
    # Example for "isn't" → "aren't" could be handled separately.
    # text = re.sub(r"\b([Hh]e|[Ss]he)\s+isn't\b", lambda m: "they aren't", text)
    
    return text



In [5]:
example = train_dataset[0]["hard_text"]
print(f"Original: {example}")
print(f"Updated: {neutralize_text(example)}")

Original: He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates
Updated: They are also the project lead of and major contributor to the open source assembler/simulator "EASy68K." They earned a master’s degree in computer science from the University of Michigan-Dearborn, where they are also an adjunct instructor. Downloads/Updates


In [6]:
bias_and_debiased_pairs = []
for idx in range(len(train_dataset)):
    biased = train_dataset[idx]["hard_text"]
    debiased = neutralize_text(biased)
    pair = (biased, debiased)
    bias_and_debiased_pairs.append(pair)


In [8]:
MODEL_NAME = "t5-small"
MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64
BATCH_SIZE = 8
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
WARMUP_STEPS = 0
OUTPUT_DIR = "../models/neutralize_model_t5"
USE_FP16 = torch.cuda.is_available()
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 50
GRADIENT_ACCUM_STEPS = 1

In [9]:
random.shuffle(bias_and_debiased_pairs)
split_idx = int(0.8 * len(bias_and_debiased_pairs))
train_data = bias_and_debiased_pairs[:split_idx]
val_data = bias_and_debiased_pairs[split_idx:]

In [10]:
class DebiasDataset(Dataset):
    def __init__(self, data_pairs: List, tokenizer: T5Tokenizer, source_max_length=64, target_max_length=64):
        self.data = data_pairs
        self.tokenizer = tokenizer
        self.source_max_length = source_max_length
        self.target_max_length = target_max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        biased_text, neutral_text = self.data[idx]
        source_text = "neutralize: " + biased_text
        target_text = neutral_text

        source_encodings = self.tokenizer(
            source_text,
            max_length=self.source_max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        target_encodings = self.tokenizer(
            target_text,
            max_length=self.target_max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = source_encodings["input_ids"].squeeze(0)
        attention_mask = source_encodings["attention_mask"].squeeze(0)
        labels = target_encodings["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }


In [11]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [12]:
train_dataset = DebiasDataset(train_data, tokenizer, MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH)
val_dataset = DebiasDataset(val_data, tokenizer, MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    fp16=USE_FP16,
    gradient_accumulation_steps=GRADIENT_ACCUM_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
)


In [ ]:
optimizer = Adafactor(
    model.parameters(),
    lr=LEARNING_RATE,
    scale_parameter=False,
    relative_step=False
)

In [ ]:
num_training_steps = (len(train_dataset) // BATCH_SIZE) * NUM_EPOCHS
lr_scheduler = get_scheduler(
    "constant", 
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=num_training_steps
)

In [ ]:
rouge = evaluate.load("rouge")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, np.ndarray) and predictions.ndim == 3:
        predictions = predictions[:, 0, :]
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    rougeL_f1 = result["rougeL"]
    return {"rougeL_f1": rougeL_f1}

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    optimizers=(optimizer, lr_scheduler),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [ ]:
trainer.train()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
test_sentence = "The doctor told the nurse that he is busy"
input_text = "neutralize: " + test_sentence
inputs = tokenizer(input_text, return_tensors="pt").to(training_args.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=MAX_TARGET_LENGTH,
        num_beams=4,
        early_stopping=True
    )
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Biased Input: ", test_sentence)
print("Neutralized Output: ", generated_text)

In [17]:
from datasets import load_dataset
train_dataset = load_dataset("LabHC/bias_in_bios", split='train[:20%]')
test_dataset = load_dataset("LabHC/bias_in_bios", split='test[:10%]')
dev_dataset = load_dataset("LabHC/bias_in_bios", split='dev[:10%]')

In [18]:
len(train_dataset)

51496